# 02 — Random Forest

Un árbol de decisión clasifica un caso mediante una sucesión de preguntas sobre sus atributos, hasta llegar a una hoja que determina la respuesta. Esas preguntas se eligen a partir de los casos de entrenamiento. Si cambian esos casos, también pueden cambiar las preguntas elegidas y, con ellas, las predicciones. Por eso, una alternativa a depender de un solo árbol consiste en construir varios y combinar sus respuestas.

Para que esa combinación resulte útil, los árboles deben aportar respuestas que no sean todas iguales. Si todos cometieran los mismos errores, reunirlos no permitiría compensarlos. Random Forest introduce variaciones aleatorias durante el aprendizaje para obtener árboles diferentes y luego reunir sus predicciones. A esta forma de combinar varios modelos se la denomina ensamble; en este caso, los modelos son árboles y su conjunto recibe el nombre de bosque. La relación entre la capacidad de cada árbol y la dependencia entre sus errores es una parte central del análisis de Breiman (2001, secciones 1 y 2).

Este cuaderno presenta esa construcción y la implementa mediante scikit-learn. Primero se explica cómo se generan los árboles y cómo se combinan sus respuestas. Después se definen las operaciones para validar los datos, preparar sus atributos, entrenar el bosque y consultar el modelo aprendido. Las celdas ejecutables importan herramientas y definen funciones; la selección de configuraciones y la evaluación de resultados se realizan en `03_experimentos.ipynb`.

### Cómo se obtienen árboles diferentes

La primera variación consiste en cambiar los casos que recibe cada árbol. A partir de los $n$ casos de entrenamiento se realizan $n$ extracciones al azar, permitiendo que un mismo caso se elija más de una vez. Por ejemplo, de los casos A, B, C y D podría obtenerse la muestra B, B, D y A: tiene el mismo tamaño, pero repite B y omite C. Este procedimiento se denomina muestreo con reemplazo o *bootstrap*, y se repite para cada árbol. Así, los árboles pueden aprender con combinaciones distintas de los casos disponibles.

La segunda variación se introduce al elegir cada pregunta del árbol. En lugar de examinar todos los atributos en cada nodo, se selecciona al azar un grupo de candidatos y se busca una división entre ellos. La selección se repite en cada nodo, por lo que un árbol no queda limitado al mismo grupo de atributos durante todo su crecimiento. Esta restricción busca evitar que un atributo dominante conduzca a todos los árboles por decisiones similares. En la implementación, `maximo_atributos` controla el tamaño de ese grupo; su valor inicial, `sqrt`, usa la raíz cuadrada de la cantidad de atributos, redondeada hacia abajo y con un mínimo de uno.

Ambas variaciones forman parte del entrenamiento, pero cumplen funciones distintas: una modifica la muestra de casos y la otra modifica los atributos candidatos. El parámetro `usar_bootstrap` permite desactivar la primera; en ese caso, cada árbol recibe todos los casos de entrenamiento. Para repetir las elecciones aleatorias se fija un número inicial, denominado semilla, mediante `semilla`. Mantener la semilla, los datos y el entorno permite reproducir el entrenamiento; no implica que la configuración elegida sea la mejor para cualquier conjunto de datos.

### Cómo aprende cada árbol

Una vez definidos los casos y los atributos candidatos, cada árbol necesita elegir cómo separar los casos de un nodo. La implementación utiliza preguntas que producen dos grupos, llamados ramas izquierda y derecha. Se prefiere una división cuando deja grupos con menor mezcla de clases que el nodo original. Esa mezcla se cuantifica mediante una medida de impureza: vale cero si todos los casos pertenecen a una misma clase.

Los criterios disponibles son Gini y entropía. Si $D$ es el conjunto de casos de un nodo y $p_c$ es la proporción que pertenece a la clase $c$, se calculan de la siguiente manera:

$$\operatorname{Gini}(D)=1-\sum_c p_c^2,\qquad H(D)=-\sum_c p_c\log_2p_c.$$

La suma recorre las clases y, en la entropía, se toma como cero el término de una clase con proporción cero. Ambos criterios permiten comparar la mezcla antes y después de una división. Para hacerlo, la impureza de cada rama se pondera por la proporción de casos que recibe. Si el nodo tiene $n$ casos y las ramas tienen $n_{izq}$ y $n_{der}$, la reducción es:

$$\Delta I=I(D)-\frac{n_{izq}}n I(D_{izq})-\frac{n_{der}}n I(D_{der}).$$

Aquí $I$ representa el criterio elegido y $D_{izq}$ y $D_{der}$ son los casos de cada rama. Una reducción mayor indica que la división deja grupos menos mezclados según ese criterio. El parámetro `criterio="gini"` utiliza Gini, mientras que `"entropy"` y `"log_loss"` utilizan entropía. Por tanto, los árboles de este bosque eligen sus divisiones con estos criterios; el cociente de ganancia utilizado por C4.5 se explica en `01_decision_tree_c45.ipynb`.

### Cómo se combinan las respuestas

Después del entrenamiento, un caso nuevo recorre cada árbol hasta llegar a una hoja. En esa hoja se obtiene la proporción de casos de entrenamiento de cada clase, contando las repeticiones de la muestra cuando se usa muestreo con reemplazo. scikit-learn interpreta esas proporciones como estimaciones de probabilidad y las promedia entre los árboles. Finalmente, el bosque devuelve la clase con el promedio más alto.

Por ejemplo, supóngase que tres árboles asignan a la clase A las probabilidades 0,6; 0,6 y 0,1. Si solo existen las clases A y B, las probabilidades de B son 0,4; 0,4 y 0,9. Los promedios son aproximadamente 0,433 para A y 0,567 para B, por lo que el bosque predice B, aunque dos árboles prefieran individualmente A. Este ejemplo ilustrativo muestra por qué promediar probabilidades puede dar una respuesta distinta de contar los votos de las clases ganadoras.

Para expresar la misma operación de forma general, se denomina $B$ a la cantidad de árboles y $P_b(c\mid x)$ a la probabilidad que el árbol $b$ asigna a la clase $c$ para el caso $x$. Su promedio se calcula como:

$$\overline P(c\mid x)=\frac1B\sum_{b=1}^{B}P_b(c\mid x).$$

La clase predicha es la que maximiza $\overline P(c\mid x)$; ante un empate, el estimador elige la primera clase según su orden interno. Agregar árboles puede estabilizar este promedio, pero también aumenta el trabajo de entrenamiento y predicción. Una mejora en los casos de prueba debe comprobarse mediante la evaluación, y no se deduce únicamente de la cantidad de árboles. Del mismo modo, una probabilidad estimada de 0,8 no garantiza por sí sola un 80 % de aciertos entre los casos que reciben ese valor; comprobar esa correspondencia requiere evaluar la calibración del modelo.

La motivación del bosque se apoya en Breiman, L. (2001), *Random Forests*, Machine Learning 45, 5–32, disponible en `docs/libros/`. La combinación de probabilidades descrita aquí corresponde al clasificador de scikit-learn utilizado por el proyecto; la definición original de Breiman combina votos de clase. Con esta distinción, las siguientes celdas concretan la preparación de datos y las operaciones necesarias para utilizar el modelo.


In [1]:
# Tablas, números y comprobaciones de tipos para las entradas del modelo.
from numbers import Integral, Real
import numpy as np
import pandas as pd
from pandas.api.types import is_bool_dtype, is_complex_dtype, is_numeric_dtype

# Componentes de la secuencia de preparación y del bosque permitido.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.utils.validation import check_is_fitted


## 1. Validación y normalización de entradas

Para llevar el procedimiento anterior al código, primero se establece cómo deben llegar los datos. Los atributos se reciben en una tabla de pandas (`DataFrame`), con una fila por caso y una columna por atributo. La tabla debe contener casos y atributos, y sus columnas deben tener nombres únicos y no nulos. Se comprueba además que los números conocidos sean reales y finitos, por lo que se rechazan los complejos y los infinitos. Estas condiciones permiten detectar entradas incompatibles antes de construir el bosque.

También se debe decidir qué hacer cuando falta un valor. La política inicial, `rechazar`, exige datos completos tanto para entrenar como para predecir. Las políticas `imputar` e `imputar_frecuente` permiten ausencias y las reemplazan durante la preparación descrita en la sección siguiente. En esta primera etapa, `normalizar_faltantes` solo representa las distintas formas de ausencia mediante `np.nan`, sobre una copia de la tabla. Esta normalización conserva los valores conocidos y todavía no calcula ningún reemplazo.

Además de revisar los datos, se validan los parámetros que controlan el bosque. La cantidad de árboles y una profundidad limitada deben ser enteros positivos; la semilla debe ser un entero entre 0 y $2^{32}-1$. Se verifican también las opciones de criterio, muestreo y cantidad de atributos, cuyos valores se detallan junto a la construcción del modelo. Las funciones de esta celda terminan sin devolver un resultado si las entradas son válidas, salvo `normalizar_faltantes`, que devuelve la copia normalizada; si una validación falla, se informa el error correspondiente.


In [2]:
def validar_entrada_bosque(caracteristicas, tratamiento_faltantes="rechazar"):
    """Valida una tabla no vacía, sus valores y la política; devuelve None."""
    # Verificar estructura y política antes de inspeccionar los datos.
    if tratamiento_faltantes not in ("rechazar", "imputar", "imputar_frecuente"):
        raise ValueError("tratamiento_faltantes debe ser rechazar, imputar o imputar_frecuente.")
    if not isinstance(caracteristicas, pd.DataFrame):
        raise TypeError("Las características deben ser un pandas.DataFrame.")
    if caracteristicas.empty:
        raise ValueError("La tabla debe contener casos y atributos.")
    if not caracteristicas.columns.is_unique or caracteristicas.columns.isna().any():
        raise ValueError("Las columnas deben tener nombres únicos y no nulos.")
    if tratamiento_faltantes == "rechazar" and caracteristicas.isna().any().any():
        raise ValueError("La política rechazar no admite valores faltantes.")

    # Admitir solo números finitos y reservar el marcador usado por la imputación.
    for columna in caracteristicas:
        serie = caracteristicas[columna]
        if is_complex_dtype(serie.dtype):
            raise ValueError("No se admiten números complejos.")
        for valor in serie.dropna():
            if isinstance(valor, complex) or (isinstance(valor, Real) and not np.isfinite(valor)):
                raise ValueError("Los números conocidos deben ser reales y finitos.")
            if tratamiento_faltantes == "imputar" and valor == "__FALTANTE__":
                raise ValueError("__FALTANTE__ está reservado para representar ausentes.")


def normalizar_faltantes(caracteristicas):
    """Devuelve una copia de la tabla con np.nan en cada posición ausente."""
    # La copia como objetos permite representar ausentes de texto y de números.
    copia = caracteristicas.astype(object)
    return copia.where(caracteristicas.notna(), np.nan)


def validar_parametros_bosque(numero_arboles, maximo_atributos, profundidad_maxima,
                             criterio, usar_bootstrap, semilla):
    """Valida los seis controles del estimador; devuelve None o lanza ValueError."""
    # Árboles, profundidad y semilla se configuran con enteros, no booleanos.
    if (not isinstance(numero_arboles, Integral) or isinstance(numero_arboles, bool)
            or numero_arboles < 1):
        raise ValueError("numero_arboles debe ser un entero positivo.")
    if profundidad_maxima is not None and (
            not isinstance(profundidad_maxima, Integral) or isinstance(profundidad_maxima, bool)
            or profundidad_maxima < 1):
        raise ValueError("profundidad_maxima debe ser None o un entero positivo.")
    if (not isinstance(semilla, Integral) or isinstance(semilla, bool)
            or not 0 <= semilla < 2**32):
        raise ValueError("semilla debe ser un entero entre 0 y 2**32 - 1.")

    # Cantidad de atributos: regla, cantidad entera o proporción.
    atributos_validos = maximo_atributos is None
    if isinstance(maximo_atributos, str):
        atributos_validos = maximo_atributos in ("sqrt", "log2")
    elif isinstance(maximo_atributos, Integral) and not isinstance(maximo_atributos, bool):
        atributos_validos = maximo_atributos >= 1
    elif isinstance(maximo_atributos, Real) and not isinstance(maximo_atributos, bool):
        atributos_validos = 0 < maximo_atributos <= 1
    if not atributos_validos:
        raise ValueError("maximo_atributos debe ser sqrt, log2, None, entero o proporción válida.")
    if criterio not in ("gini", "entropy", "log_loss"):
        raise ValueError("criterio debe ser gini, entropy o log_loss.")
    if not isinstance(usar_bootstrap, bool):
        raise ValueError("usar_bootstrap debe ser True o False.")


## 2. Preparación de atributos, construcción y entrenamiento

Una vez validadas las entradas, los atributos deben convertirse a la representación numérica que utiliza el bosque. Para ello se distinguen las columnas numéricas de las categóricas a partir de sus tipos originales. Las numéricas mantienen sus valores conocidos, sin escalado, porque las preguntas del árbol comparan valores con umbrales. Las categóricas requieren una transformación que represente sus categorías sin asignarles un orden numérico artificial. Por esa razón, una categoría codificada con números debe convertirse previamente a texto o al tipo `category`; los booleanos también se tratan como categorías.

La transformación categórica crea una columna indicadora por cada categoría aprendida en entrenamiento. Por ejemplo, un atributo con valores rojo y azul se representa con dos columnas: rojo pasa a (1, 0) y azul a (0, 1), si ese es el orden de las columnas. Esta representación se denomina codificación *one-hot*. Si al predecir aparece una categoría nueva, se colocan ceros en todas las columnas correspondientes a ese atributo. Como el bosque recibe la tabla transformada, `maximo_atributos` se aplica a estas columnas indicadoras y a las columnas numéricas, no a la cantidad original de atributos.

Antes de codificar, se resuelven las ausencias según la política seleccionada. Reemplazar un dato faltante por un valor definido se denomina imputación. Cuando ese valor se calcula a partir de los datos, se utilizan exclusivamente los casos de entrenamiento. De este modo, los casos de prueba no intervienen en la elección de los reemplazos. Las tres políticas disponibles son:

| Política | Tratamiento de los faltantes |
|---|---|
| `rechazar` | Informa un error si hay ausencias. Es el valor inicial. |
| `imputar_frecuente` | En columnas numéricas usa la mediana; en categóricas, la categoría más frecuente, denominada moda. Ambos valores se aprenden en entrenamiento. En empates de frecuencia se utiliza el orden que aplica el imputador. |
| `imputar` | En columnas numéricas usa la mediana de entrenamiento; en categóricas, el texto reservado `__FALTANTE__`, que no puede aparecer como categoría real. |

Si una columna de entrenamiento está completamente vacía, no se puede calcular su mediana o su moda. Para conservarla, los imputadores usan `keep_empty_features=True`. En ese caso, las columnas numéricas reciben 0; las categóricas reciben 0 con `imputar_frecuente` o el marcador reservado con `imputar`. Esta decisión mantiene explícito el tratamiento de ausencias, sin depender de que el bosque de una versión de la biblioteca las admita directamente.

La normalización, la preparación de atributos y el bosque se reúnen en una única secuencia de operaciones, denominada *pipeline* en scikit-learn. Durante el entrenamiento, esa secuencia aprende los reemplazos que correspondan y las categorías, transforma la tabla y ajusta los árboles. Durante la predicción aplica la preparación ya aprendida antes de consultar esos mismos árboles. Así se conserva el significado de las columnas entre ambas etapas. Si se entrena un modelo nuevo, los reemplazos y las categorías se aprenden nuevamente con sus datos de entrenamiento.

### Configuración del modelo

Con la preparación definida, los parámetros permiten elegir cómo construir el bosque. La tabla siguiente relaciona los nombres usados por las funciones del proyecto con los nombres de scikit-learn. Los valores iniciales describen la configuración aplicada cuando no se proporciona otra. La comparación y la selección de configuraciones se realizan en el cuaderno de experimentos.

| Parámetro del proyecto | Parámetro del estimador | Efecto y valor inicial |
|---|---|---|
| `numero_arboles` | `n_estimators` | Cantidad de árboles: 100. |
| `maximo_atributos` | `max_features` | Cantidad de columnas candidatas por nodo: `sqrt`. Admite también `log2`, `None` para todas, entero positivo o proporción mayor que 0 y menor o igual que 1. |
| `profundidad_maxima` | `max_depth` | Profundidad máxima, contando la raíz como profundidad 0: `None`, sin límite explícito. |
| `criterio` | `criterion` | Medida de impureza: `gini`, `entropy` o `log_loss`; inicialmente `gini`. |
| `usar_bootstrap` | `bootstrap` | Activa el muestreo con reemplazo: `True`. |
| `semilla` | `random_state` | Número inicial para las elecciones aleatorias: 42. |
| `tratamiento_faltantes` | Preparación del pipeline | `rechazar`, `imputar` o `imputar_frecuente`; inicialmente `rechazar`. |

Además de estos parámetros configurables, se fijan dos casos como mínimo para intentar dividir un nodo y uno como mínimo por hoja. Estos mínimos no obligan a dividir si no existe una separación válida. El cálculo utiliza un único proceso (`n_jobs=1`) para mantener una condición común en las mediciones de tiempo. Aun con esa condición, los tiempos pueden variar entre ejecuciones y se registran en el cuaderno de experimentos.

### De la configuración al modelo aprendido

La función `crear_bosque` recibe la tabla de atributos y los parámetros, y devuelve el pipeline sin entrenar. En ese momento recuerda los nombres, el orden y los tipos numéricos de las columnas, además de la configuración elegida. Todavía no aprende reemplazos, categorías ni divisiones. El nombre `crear_modelo` es un alias de esta misma función, conservado por compatibilidad con el resto del proyecto.

Para aprender a clasificar se utiliza `entrenar_bosque`, que recibe los atributos y una segunda tabla con las respuestas conocidas. Esta segunda tabla debe contener solo la columna `objetivo`, sin valores ausentes y con el mismo índice y orden que la tabla de atributos. La función construye el pipeline con los parámetros recibidos, lo ajusta con ambas tablas y devuelve el modelo entrenado. Ese modelo queda preparado para la operación de predicción que se define a continuación.


In [3]:
def crear_bosque(caracteristicas, numero_arboles=100, maximo_atributos="sqrt",
                 profundidad_maxima=None, criterio="gini", usar_bootstrap=True,
                 semilla=42, tratamiento_faltantes="rechazar"):
    """Construye un pipeline sin ajustar y registra columnas, tipos y parámetros.

    caracteristicas fija el esquema de entrada. Los demás argumentos configuran
    el bosque y la política de ausencias; el valor inicial rechaza faltantes.
    """
    # Validar la configuración y recordar los tipos originales de las columnas.
    validar_entrada_bosque(caracteristicas, tratamiento_faltantes)
    validar_parametros_bosque(numero_arboles, maximo_atributos, profundidad_maxima,
                             criterio, usar_bootstrap, semilla)
    columnas_numericas = [columna for columna in caracteristicas
                          if is_numeric_dtype(caracteristicas[columna].dtype)
                          and not is_bool_dtype(caracteristicas[columna].dtype)]
    columnas_categoricas = [columna for columna in caracteristicas
                            if columna not in columnas_numericas]

    # Preparar cada tipo de columna; la imputación se ajustará solo al entrenar.
    pasos_categoricos = []
    preparacion_numerica = "passthrough"
    if tratamiento_faltantes in ("imputar", "imputar_frecuente"):
        # La moda usa la categoría más frecuente; el marcador conserva la ausencia como señal.
        imputador_categorico = (SimpleImputer(strategy="most_frequent", keep_empty_features=True)
            if tratamiento_faltantes == "imputar_frecuente" else
            SimpleImputer(strategy="constant", fill_value="__FALTANTE__", keep_empty_features=True))
        pasos_categoricos.append(("imputar", imputador_categorico))
        preparacion_numerica = SimpleImputer(strategy="median", keep_empty_features=True)
    pasos_categoricos.append(("codificar", OneHotEncoder(
        handle_unknown="ignore", sparse_output=False)))
    preparador = ColumnTransformer([
        ("categoricas", Pipeline(pasos_categoricos), columnas_categoricas),
        ("numericas", preparacion_numerica, columnas_numericas),
    ])

    # Unir la preparación y los árboles en una sola secuencia reutilizable.
    modelo = Pipeline([
        ("normalizar", FunctionTransformer(normalizar_faltantes, feature_names_out="one-to-one")),
        ("preprocesador", preparador),
        ("random_forest", RandomForestClassifier(
            n_estimators=numero_arboles, max_features=maximo_atributos,
            max_depth=profundidad_maxima, criterion=criterio,
            bootstrap=usar_bootstrap, random_state=semilla, n_jobs=1,
            min_samples_split=2, min_samples_leaf=1)),
    ])

    # Guardar el contrato original y los valores efectivos para registrar la corrida.
    modelo.columnas_entrada_ = caracteristicas.columns.copy()
    modelo.columnas_numericas_ = columnas_numericas
    modelo.configuracion_ = dict(
        numero_arboles=numero_arboles, maximo_atributos=maximo_atributos,
        profundidad_maxima=profundidad_maxima, criterio=criterio,
        usar_bootstrap=usar_bootstrap, semilla=semilla,
        tratamiento_faltantes=tratamiento_faltantes,
        minimo_muestras_division=2, minimo_muestras_hoja=1, procesos=1)
    return modelo


def entrenar_bosque(caracteristicas, objetivos, **parametros):
    """Ajusta y devuelve el pipeline con características y objetivos alineados.

    objetivos debe tener solo la columna objetivo, sin ausencias. Los parámetros
    se transmiten a crear_bosque; la preparación se aprende solo aquí.
    """
    # La respuesta correcta permanece separada de las características.
    if not isinstance(objetivos, pd.DataFrame):
        raise TypeError("Los objetivos deben ser un pandas.DataFrame.")
    if (not isinstance(caracteristicas, pd.DataFrame)
            or list(objetivos.columns) != ["objetivo"]
            or not caracteristicas.index.equals(objetivos.index)):
        raise ValueError("Se necesitan características y objetivo con el mismo índice y orden.")
    if objetivos.isna().any().any():
        raise ValueError("Las respuestas correctas no deben contener faltantes.")

    # El estimador recibe un vector, conservando afuera el DataFrame original.
    modelo = crear_bosque(caracteristicas, **parametros)
    modelo.fit(caracteristicas, objetivos["objetivo"])
    return modelo


## 3. Predicción con el modelo entrenado

Una vez aprendido el modelo, `crear_predictor` permite utilizarlo mediante una función que recibe nuevos casos. Primero comprueba que el bosque esté entrenado y luego devuelve la función `predecir(caracteristicas)`. Cada consulta debe conservar los nombres y el orden de las columnas de entrenamiento. Además, una columna que se aprendió como numérica debe seguir conteniendo números, y las ausencias deben respetar la política elegida al construir el modelo. Estas comprobaciones evitan aplicar las transformaciones aprendidas a una tabla con otro significado.

Si la consulta es válida, el pipeline aplica su preparación y el bosque combina las probabilidades como se explicó al comienzo. La función devuelve una tabla con una única columna, `prediccion`, y una fila por caso. Se conservan el orden y los índices de entrada, incluso si hay índices duplicados, y no se modifica la tabla consultada. La consulta reutiliza el modelo aprendido y no vuelve a entrenarlo. Esta salida permite que el cuaderno de experimentos compare las clases predichas con las respuestas conocidas y calcule las métricas correspondientes.


In [4]:
def crear_predictor(modelo):
    """Recibe un pipeline ajustado y devuelve una función de predicción.

    La función exige el esquema original y devuelve una tabla prediccion con
    los mismos índices y el mismo orden, sin modificar la consulta.
    """
    # Comprobar que el bosque aprendió antes de aceptar consultas.
    check_is_fitted(modelo.named_steps["random_forest"])
    columnas = modelo.columnas_entrada_.copy()
    tratamiento = modelo.configuracion_["tratamiento_faltantes"]

    def predecir(caracteristicas):
        """Clasifica casos conservando su índice y orden, sin volver a entrenar."""
        # Mantener la misma estructura y el significado de los atributos numéricos.
        validar_entrada_bosque(caracteristicas, tratamiento)
        if not caracteristicas.columns.equals(columnas):
            raise ValueError("La consulta debe conservar columnas y orden de entrenamiento.")
        for columna in modelo.columnas_numericas_:
            if any(not isinstance(valor, Real) or isinstance(valor, bool)
                   for valor in caracteristicas[columna].dropna()):
                raise ValueError(f"El atributo {columna!r} debe conservar su tipo numérico.")

        # El pipeline aplica su preparación aprendida y entrega una etiqueta por caso.
        etiquetas = modelo.predict(caracteristicas)
        return pd.DataFrame({"prediccion": etiquetas}, index=caracteristicas.index.copy())

    return predecir


# Nombre compatible para construir exactamente el mismo pipeline.
crear_modelo = crear_bosque
